# CE-clean-only full-scale baseline — crop1.5 (Seed 105)

Train **one** new binary MiniFASNetV2:  with **TRAIN_SEED=105**.
Total loss = CE_clean; no worst-view loss, spectral selection or auxiliary loss/forward. Historical C_FULL includes L_aux and is a different baseline.

Matched notebook18 full-scale controls: full eligible CelebA Train, full Test-as-Val selection/calibration, full LCC train/dev/eval/combined benchmarks, fixed architecture/crop factor 1.5, batch 256, 12 epochs (milestones [4,8], gamma 0.2), SGD lr=0.005, momentum=0.9, weight_decay=5e-4, data seed 43.

Tự động nhận diện đường dẫn Dataset trên **Kaggle** (hỗ trợ mọi slug phổ biến , , ). Tự động đánh giá đầy đủ **CelebA Test**, **4-split LCC FASD** và **CASIA-FASD** (Zero-shot + Calibrated), xuất file ZIP hoàn chỉnh để tải về chỉ với 1 click.

In [1]:
# Edit Kaggle paths here. Full official split manifests are in RESOURCE_ROOT.
import sys, os
from pathlib import Path

def find_path(candidates, default):
    for p in candidates:
        if Path(p).exists():
            return str(p)
    return str(default)

TRAIN_SEED = 105  # Seed 105; DATA_SEED remains 43.
DATA_SEED = 43
RUN_FILTER = None  # None or ["CE_CLEAN_FULL_crop15"].
RESUME = True
NUM_WORKERS = 4
AMP = True
BATCH_SIZE = 256
MAX_EPOCHS = 12
MILESTONES = [4, 8]
GAMMA = .2
PATIENCE_START_EPOCH = 9
EARLY_STOP_PATIENCE = 3
WARMUP_EPOCHS = 1
CSMR_PRESERVE_DC = True
SELECTION_CHUNK_SIZE = 32
CROP_SCALE = 1.5
TRAIN_CROP_FACTOR = 1.5
EVAL_CROP_FACTOR = 1.5
LOG_EVERY_BATCHES = 25
INPUT_SIZE = 80
METRIC_TIE_EPS = 1e-12

RESOURCE_ROOT = find_path([
    '/kaggle/input/datasets/locngodev/full-resources-pooled/minifasnet_binary_crossdomain_resources_v2_lcc_pooled',
    '/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled',
    '/kaggle/input/datasets/quetv5/full-resources/minifasnet_binary_crossdomain_resources_v2_lcc_pooled',
    '/kaggle/input/full-resources-pooled/minifasnet_binary_crossdomain_resources_v2_lcc_pooled',
    '/kaggle/input/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled',
    '/kaggle/input/full-resources/minifasnet_binary_crossdomain_resources_v2_lcc_pooled',
    '/content/full_resources/minifasnet_binary_crossdomain_resources_v2_lcc_pooled',
    '/content/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
], '/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled')

CELEBA_ROOT = find_path([
    '/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof',
    '/kaggle/input/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof',
    '/kaggle/input/celeba-spoof/CelebA_Spoof_/CelebA_Spoof',
    '/content/celeba_spoof/CelebA_Spoof_/CelebA_Spoof'
], '/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof')

LCC_ROOT = find_path([
    '/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD',
    '/kaggle/input/lcc-fasd/LCC_FASD',
    '/content/lcc_fasd/LCC_FASD',
    '/content/LCC_FASD'
], '/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD')

MINIFASNET_SOURCE_PATH = find_path([
    '/kaggle/input/datasets/locngodev/minifasnet/MiniFASNet.py',
    '/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py',
    '/kaggle/input/datasets/quetv5/minifasnet/MiniFASNet.py',
    '/kaggle/input/minifasnet/MiniFASNet.py',
    '/content/minifasnet/MiniFASNet.py',
    '/content/MiniFASNet.py'
], '/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py')

MINIFASNET_CHECKPOINT_PATH = find_path([
    '/kaggle/input/datasets/locngodev/minifasnet/2.7_80x80_MiniFASNetV2.pth',
    '/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth',
    '/kaggle/input/datasets/quetv5/minifasnet/2.7_80x80_MiniFASNetV2.pth',
    '/kaggle/input/minifasnet/2.7_80x80_MiniFASNetV2.pth',
    '/content/minifasnet/2.7_80x80_MiniFASNetV2.pth',
    '/content/2.7_80x80_MiniFASNetV2.pth'
], '/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth')

CASIA_ROOT = find_path([
    '/kaggle/input/datasets/minhnh2107/casiafasd',
    '/kaggle/input/casiafasd',
    '/kaggle/input/casia-fasd',
    '/content/casiafasd',
    '/content/casia_fasd'
], '/kaggle/input/datasets/minhnh2107/casiafasd')

SCRFD_MODEL_PATH = find_path([
    '/kaggle/input/datasets/quetv5/scrfd-models/det_500m.onnx',
    '/kaggle/input/scrfd-model/det_500m.onnx',
    '/content/scrfd/det_500m.onnx',
    '/content/det_500m.onnx'
], '/kaggle/input/datasets/locngodev/scrfd-model/det_500m.onnx')

OUTPUT_ROOT = f'/kaggle/working/21_fullscale_CE_clean_baseline_crop15_s{TRAIN_SEED}'

RUNS = {'CE_CLEAN_FULL_crop15': {'method': 'CLEAN', 'objective': 'CE_clean only', 'warmup_epochs': 0,
 'crop_factor': 1.5, 'centers': None, 'sigma': None, 'clean_ce_weight': 1., 'spectral_ce_weight': 0., 'aux_loss_weight': 0.}}

def select_runs(run_filter):
    if run_filter is not None and (not isinstance(run_filter, (list, tuple)) or not run_filter or len(set(run_filter)) != len(run_filter) or set(run_filter) - set(RUNS)):
        raise ValueError('RUN_FILTER must be None or ["CE_CLEAN_FULL_crop15"]')
    return [name for name in RUNS if run_filter is None or name in run_filter]

CALIBRATE_CASIA_THRESHOLD = True  # train-only fitting; source results retained
INSTALL_DETECTOR_DEPS = True
ONNXRUNTIME_GPU_SPEC = 'onnxruntime-gpu==1.26.0'
MIN_VALID_FRAMES = 4
DETECTION_LOG_EVERY = 100
DETECTOR_SIZES = [(640, 640), (480, 480), (320, 320)]
DETECTOR_CONFIDENCE = .5


In [2]:
from pathlib import Path
# SETUP — import PyTorch first to expose its CUDA/cuDNN libraries.
import os,sys,subprocess,random,time,gc,importlib.util
from importlib.metadata import version,PackageNotFoundError
import torch
print('PyTorch:',torch.__version__, 'CUDA:',torch.version.cuda, 'GPU:',torch.cuda.is_available(),flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Enable Kaggle Settings > Accelerator > GPU and restart the session.')
def installed(name):
    try:return version(name)
    except PackageNotFoundError:return None
if INSTALL_DETECTOR_DEPS:
    ready=(installed('insightface')=='2.0' and installed('onnxruntime-gpu')=='1.26.0' and installed('onnxruntime') is None)
    if not ready:
        if 'onnxruntime' in sys.modules or 'insightface' in sys.modules:
            raise RuntimeError('Restart the Kaggle session before changing detector dependencies.')
        subprocess.check_call([sys.executable,'-m','pip','install','-q','insightface==2.0'])
        subprocess.check_call([sys.executable,'-m','pip','uninstall','-y','onnxruntime','onnxruntime-gpu'])
        subprocess.check_call([sys.executable,'-m','pip','install','-q',ONNXRUNTIME_GPU_SPEC])
import cv2,numpy as np,pandas as pd,json,re,math,zipfile,types
from torch import nn
from sklearn.metrics import roc_curve,roc_auc_score
from IPython.display import display,FileLink
import matplotlib.pyplot as plt
import onnxruntime as ort
from insightface import model_zoo
if hasattr(ort,'preload_dlls'):ort.preload_dlls()
print('ONNX Runtime:',ort.__version__,'providers:',ort.get_available_providers(),flush=True)
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('CUDAExecutionProvider unavailable; restart and rerun the setup cell.')
random.seed(TRAIN_SEED);np.random.seed(TRAIN_SEED);torch.manual_seed(TRAIN_SEED);torch.cuda.manual_seed_all(TRAIN_SEED)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
cv2.setNumThreads(1)
device=torch.device('cuda')
Path(OUTPUT_ROOT).mkdir(parents=True,exist_ok=True)
for path in [Path(SCRFD_MODEL_PATH),Path(CASIA_ROOT)/'train_img/train_img/color',Path(CASIA_ROOT)/'test_img/test_img/color']:
    if not path.exists():raise FileNotFoundError(path)

# Check the real detector session before spending time on full-scale training.
preflight_detector=model_zoo.get_model(str(SCRFD_MODEL_PATH),providers=['CUDAExecutionProvider','CPUExecutionProvider'])
if preflight_detector is None:raise RuntimeError('Could not load mounted SCRFD model')
preflight_detector.prepare(ctx_id=0,input_size=DETECTOR_SIZES[0],det_thresh=DETECTOR_CONFIDENCE)
if 'CUDAExecutionProvider' not in preflight_detector.session.get_providers():
    raise RuntimeError('SCRFD CUDA session failed; fix detector dependencies before training')
print('SCRFD preflight providers:',preflight_detector.session.get_providers(),flush=True)
del preflight_detector;gc.collect();torch.cuda.empty_cache()


PyTorch: 2.11.0+cu128 CUDA: 12.8 GPU: True
Found existing installation: onnxruntime 1.31.1
Uninstalling onnxruntime-1.31.1:
  Successfully uninstalled onnxruntime-1.31.1


ONNX Runtime: 1.26.0 providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
SCRFD preflight providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']


In [3]:

import hashlib, time, shutil, importlib.util, inspect, json, math, os, random, zipfile
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display

_BBOX_RNG=random.Random(TRAIN_SEED+500000)
AUG_SEED=TRAIN_SEED
BBOX_SEED=TRAIN_SEED+500000
GAIN_SEED=TRAIN_SEED+11000
GAIN_RANGE=(.65,.90)
SPLITS=('training','development','evaluation')
EXPECTED_LCC={'training':8299,'development':2948,'evaluation':7580,'combined':18827}
if RUN_FILTER is not None and (not isinstance(RUN_FILTER,(list,tuple)) or not RUN_FILTER or len(set(RUN_FILTER))!=len(RUN_FILTER) or set(RUN_FILTER)-set(RUNS)):
    raise ValueError('RUN_FILTER contains an unknown or duplicate run ID')
if NUM_WORKERS<1:raise ValueError('NUM_WORKERS must be >= 1')
if (BATCH_SIZE,MAX_EPOCHS,MILESTONES,GAMMA)!=(256,12,[4,8],.2) or type(TRAIN_SEED) is not int or TRAIN_SEED not in (100,101,102,105):raise RuntimeError('Frozen fullscale recipe or approved seed changed')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for path in (source,pretrained,root/'manifests/crossdomain_train_full.csv',
             root/'manifests/crossdomain_celeba_source_dev_full.csv',root/'bbox/celeba_bbox_cache_full.json',
             root/'labels/celeba_auxiliary_labels.csv'):
    if not path.is_file():raise FileNotFoundError(path)
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def save_torch(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(value,tmp);os.replace(tmp,path)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()
def state_sha(state):
    h=hashlib.sha256()
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous();h.update(k.encode());h.update(str(v.dtype).encode());h.update(str(tuple(v.shape)).encode());h.update(v.numpy().tobytes())
    return h.hexdigest()
def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))

# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}



def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)


def restore_rng(state,gain_rng):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda']);gain_rng.set_state(state['gain']);_BBOX_RNG.setstate(state['bbox'])
def bbox_jitter(rec):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if _BBOX_RNG.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=_BBOX_RNG.uniform(.95,1.05)
        cx+=_BBOX_RNG.uniform(-.05,.05)*base;cy+=_BBOX_RNG.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]


def load_frame(name):return pd.read_csv(root/'manifests'/(name+'.csv'),keep_default_na=False)
train=load_frame('crossdomain_train_full')
test=load_frame('crossdomain_celeba_source_dev_full')
val=test.loc[test.usable_for_pad.astype(bool)].copy()  # Full official Test-as-Val selection set.
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records']
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
if (len(train),len(test),len(val))!=(484561,67170,66379):raise RuntimeError('Unexpected full official populations')
if train.sample_key.duplicated().any() or test.sample_key.duplicated().any() or set(train.sample_key)&set(test.sample_key):raise RuntimeError('Train/Test sample overlap')
for frame in (train,val,test):
    if not set(frame.sample_key)<=set(records) or not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing cache/aux entry')
for frame in (train,val):
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:raise RuntimeError('Unusable full split')
# Candidate accounting uses existing official metadata, with the unchanged 00c eligibility rule.
def train_exclusion_reason(key):
    rec=records[key]
    if rec['status'] in {'SUSPICIOUS','DET_FAIL','UNREADABLE'}:return rec['status']
    try:require_record(rec,key)
    except RuntimeError:return 'INVALID_BBOX'
    x1,y1,x2,y2=map(float,rec['bbox_xyxy'])
    return 'TOO_SMALL' if min(x2-x1,y2-y1)<48 else ''
train_candidates=aux.loc[aux.official_split=='train'].reset_index()
train_candidates['exclusion_reason']=[train_exclusion_reason(k) for k in train_candidates.sample_key]
if len(train_candidates)!=494405 or set(train_candidates.loc[train_candidates.exclusion_reason=='','sample_key'])!=set(train.sample_key):raise RuntimeError('Full train membership differs from official eligibility; no sampling allowed')
if set(aux.loc[aux.official_split=='test'].index)!=set(test.sample_key):raise RuntimeError('Test manifest is not the full official Test')
def population_counts(frame,three_col='three_class_label',attack_col='attack_code'):
    classes={str(k):int(v) for k,v in frame[three_col].value_counts().sort_index().items()}
    return {'n':len(frame),'three_class_counts':classes,'real_n':classes.get('0',0),'attack_n':len(frame)-classes.get('0',0),
            'spoof_type_counts':{str(k):int(v) for k,v in frame[attack_col].value_counts().sort_index().items()}}
DATA_COUNTS={'train_candidates':population_counts(train_candidates,attack_col='spoof_type'),
 'train_usable':population_counts(train),'train_excluded':len(train_candidates)-len(train),
 'train_exclusion_reasons':{str(k):int(v) for k,v in train_candidates.loc[train_candidates.exclusion_reason!='','exclusion_reason'].value_counts().items()},
 'test_candidates':population_counts(test),'test_usable':population_counts(val),'test_unscored':len(test)-len(val),
 'train_test_identity_overlap':sorted(set(train.subject_id.astype(str))&set(test.subject_id.astype(str))),
 'identity_overlap_policy':'Official membership retained; no subject removal or new split',
 'sampling':'None; full official eligible Train and full official Test-as-Val', 'data_seed':DATA_SEED}
lcc_frames={};lcc_records={}
for split in ('training','development','evaluation','combined'):
    name='lcc_official_evaluation_full' if split=='evaluation' else 'lcc_'+split+'_full'
    cache_name='lcc_official_evaluation_bbox_cache' if split=='evaluation' else 'lcc_'+split+'_bbox_cache'
    path=root/'manifests'/(name+'.csv');cache=root/'bbox'/(cache_name+'.json')
    if not path.is_file() or not cache.is_file():raise FileNotFoundError((path,cache))
    frame=pd.read_csv(path,keep_default_na=False)
    if len(frame)!=EXPECTED_LCC[split] or frame.path.duplicated().any():raise RuntimeError('Unexpected LCC population: '+split)
    recs=read_json(cache)['records'] if split!='combined' else None
    if recs is not None and set(frame.path)!=set(recs):raise RuntimeError('LCC manifest/cache keys differ: '+split)
    lcc_frames[split]=frame;lcc_records[split]=recs
ordered_pool=pd.concat([lcc_frames[s].assign(lcc_split=s) for s in SPLITS],ignore_index=True)
if not lcc_frames['combined'].equals(ordered_pool):raise RuntimeError('Combined LCC manifest differs from physical splits')
if not torch.cuda.is_available():raise RuntimeError('Kaggle CUDA GPU required for the frozen fullscale batch size')
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source)
module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)
out.mkdir(parents=True,exist_ok=True);(out/'runs').mkdir(exist_ok=True)
(out/'manifests').mkdir(exist_ok=True)
MANIFEST_SOURCES={'full_train.csv':root/'manifests/crossdomain_train_full.csv',
                  'full_test.csv':root/'manifests/crossdomain_celeba_source_dev_full.csv'}
MANIFEST_HASHES={f:sha(p) for f,p in MANIFEST_SOURCES.items()}
for filename,path in MANIFEST_SOURCES.items():shutil.copyfile(path,out/'manifests'/filename)
train_candidates.to_csv(out/'manifests'/'full_train_candidates.csv',index=False)
write_json(out/'manifest_sha256.json',MANIFEST_HASHES)
write_json(out/'dataset_counts.json',DATA_COUNTS)
print('GPU:',torch.cuda.get_device_name(0),'Full Train:',len(train),'Full Test-as-Val usable/candidate:',len(val),len(test),'LCC:',{s:len(f) for s,f in lcc_frames.items()},flush=True)

def init_model(spec):
    seed_all(TRAIN_SEED);model=PADNet(spec['copied']);audit={'initialization':spec['init'],'checkpoint_sha256':sha(pretrained),'source_sha256':sha(source)}
    if spec['init']=='pretrained':
        raw=torch.load(pretrained,map_location='cpu',weights_only=True)
        if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):raise RuntimeError('Official checkpoint format mismatch')
        stripped=OrderedDict((k[7:],v) for k,v in raw.items())
        if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official checkpoint architecture mismatch')
        target=model.trunk.state_dict();compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
        bad=[k for k,v in compatible.items() if k not in target or tuple(v.shape)!=tuple(target[k].shape)]
        absent=[k for k in target if k not in compatible and k!='prob.weight']
        if bad or absent:raise RuntimeError(f'Pretrained key mismatch: {bad[:5]} {absent[:5]}')
        missing,unexpected=model.trunk.load_state_dict(compatible,strict=False)
        if unexpected or (spec['copied'] and missing) or (not spec['copied'] and set(missing)!={'prob.weight'}):raise RuntimeError('Pretrained load mismatch')
        # Reinitialize historical classifier too; copied heads were initialized by seed 100.
        if not spec['copied']:model.trunk.prob.reset_parameters()
        audit.update({'loaded_keys':sorted(compatible),'skipped_keys':['prob.weight'],'missing_keys':list(missing),'unexpected_keys':list(unexpected)})
    else:audit.update({'loaded_keys':[],'skipped_keys':[],'missing_keys':[],'unexpected_keys':[]})
    for head in (model.spoof_type_head,model.lighting_head,model.attributes_head):
        head.requires_grad_(False)
    audit['objective']='CE_clean only; auxiliary modules frozen and never forwarded'
    audit['initial_state_sha256']=state_sha(model.state_dict())
    with torch.no_grad():
        model.eval();assert model(torch.zeros(2,3,80,80)).shape==(2,2 if spec['copied'] else 3)
    return model,audit

INITIAL_AUDITS={}
for name in RUNS:
    initial_model,initial_audit=init_model({'init':'pretrained','copied':True})
    INITIAL_AUDITS[name]=initial_audit
    del initial_model
INITIAL_HASHES={name:audit['initial_state_sha256'] for name,audit in INITIAL_AUDITS.items()}
if len(set(INITIAL_HASHES.values()))!=1:raise RuntimeError('Initial-state SHA differs')
INITIAL_STATE_SHA=next(iter(INITIAL_HASHES.values()))
write_json(out/'initialization_fairness.json',INITIAL_AUDITS)

class CelebaData(Dataset):
    def __init__(self,frame,training=False,copied=False):self.rows=frame.reset_index(drop=True);self.training=training;self.copied=copied
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        patch=mini_crop_bgr(image,bbox_jitter(rec) if self.training else rec['bbox_xyxy'])
        if self.training:patch=copied_aug_bgr(patch) if self.copied else mild_aug(patch)
        # Legacy batch shape retained; auxiliary targets are not read or used.
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),0,0,torch.zeros(40),key

class LCCData(Dataset):
    def __init__(self,split):
        self.split=split;self.rows=lcc_frames[split].loc[lcc_frames[split].usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[self.split][key],key)
        path=(Path(LCC_ROOT)/'LCC_FASD_evaluation'/key) if self.split=='evaluation' else Path(LCC_ROOT)/key
        image=cv2.imread(str(path),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(path)
        return tensor_of(mini_crop_bgr(image,rec['bbox_xyxy'])),int(row.label),key

def loader(ds,batch=BATCH_SIZE,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    def worker_init(wid):
        global _BBOX_RNG
        s=TRAIN_SEED+epoch*1000+wid;random.seed(s);np.random.seed(s);torch.manual_seed(s)
        _BBOX_RNG=random.Random(s+500000)
        if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(s)
    return DataLoader(ds,batch_size=batch,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,drop_last=False,generator=gen,worker_init_fn=worker_init)

def infer(model,dl,lcc_mode=False):
    model.eval();keys=[];ys=[];zs=[];started=time.monotonic()
    print('[Inference] Starting',len(dl.dataset),'scored images',flush=True)
    with torch.no_grad():
        for step,batch in enumerate(dl,1):
            if lcc_mode:x,y,k=batch
            else:x,y,_,_,_,_,k=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy())
            if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(dl):print(f'[Inference] batch {step}/{len(dl)}, elapsed={time.monotonic()-started:.1f}s',flush=True)
    if not zs:raise RuntimeError('No scorable evaluation samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs)




def training_terms(model,x,binary,spoof,lighting,attrs,spec,masks,gain_rng,active):
    # A single clean binary forward; no auxiliary outputs or spectral candidates.
    with torch.cuda.amp.autocast(enabled=use_amp):
        logits=model(x)
        loss=F.cross_entropy(logits,binary)
    clean_ce=float(loss.detach())
    return loss,{'train_total_loss':clean_ce,'total_loss':clean_ce,'clean_ce':clean_ce,
                 'spectral_ce':0.,'worst_ce':0.,'aux_loss':0.,'harmful_count':0.,
                 'harmful_fraction':0.,'harmful_ce':0.},Counter(),0.


def snapshot_rng(gain_rng):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all(),'gain':gain_rng.get_state(),'bbox':_BBOX_RNG.getstate()}

def run_config(name,spec):
    return {'run_id':name,'spec':spec,'resource_root':str(root),'train_manifest':'full_train.csv',
      'val_manifest':'full_test.csv','selection_set':'Full official CelebA Test-as-Val; reused selection set, not untouched Test','manifest_sha256':MANIFEST_HASHES,
      'train_crop_factor':1.5,'val_crop_factor':1.5,'test_crop_factor':1.5,'lcc_crop_factor':1.5,'data_seed':DATA_SEED,
      'train_seed':TRAIN_SEED,'initial_state_sha256':INITIAL_STATE_SHA,
      'base_order':'torch Generator(TRAIN_SEED+epoch)','worker_seed':'TRAIN_SEED+epoch*1000+worker_id',
      'bbox_rng_seed':'worker_seed+500000','gain_rng_seed':GAIN_SEED,
      'pretrained_checkpoint':str(pretrained),'model_source':str(source),
      'celeba_root':CELEBA_ROOT,'lcc_root':LCC_ROOT,'num_workers':NUM_WORKERS,
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'milestones':MILESTONES,'gamma':GAMMA,
      'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'warmup_epochs':spec['warmup_epochs'],'patience_start_epoch':PATIENCE_START_EPOCH,'early_stop_patience':EARLY_STOP_PATIENCE,
      'loss':{'CE_clean':1.,'CE_worst':0.,'L_aux':0.},'spectral':None,
      'auxiliary_heads':'state-dict compatibility only; frozen, excluded from optimizer, never forwarded',
      'casia_root':CASIA_ROOT,'scrfd_model_path':SCRFD_MODEL_PATH,'casia_train_threshold_calibration':CALIBRATE_CASIA_THRESHOLD,
      'input':'80x80 BGR float [0,1] crop1.5','augmentation_api':AUGMENTATION_API,
      'albumentations_version':A.__version__,'amp':use_amp}

def prediction_frame(frame,keys,y,z,threshold,keycol):
    scores=pad_score(z)
    scored=pd.DataFrame({keycol:keys,'pad_logit_score':scores,'pred_real':scores>=threshold,
                         'logit_0':z[:,0],'logit_1':z[:,1]})
    result=frame.merge(scored,on=keycol,how='left',validate='one_to_one',sort=False)
    valid=frame.usable_for_pad.astype(bool)
    if result.loc[valid,'pad_logit_score'].isna().any() or result.loc[~valid,'pad_logit_score'].notna().any():
        raise RuntimeError('Prediction accounting differs from frozen candidates')
    result['locked_threshold']=threshold
    return result

def evaluate_lcc(model,folder,locked):
    scored_predictions=[]
    for split in SPLITS:
        frame=lcc_frames[split]
        keys,y,z=infer(model,loader(LCCData(split)),True)
        expected=frame.loc[frame.usable_for_pad.astype(bool),'path'].astype(str).tolist()
        if keys!=expected:raise RuntimeError('LCC scored order differs: '+split)
        pred=prediction_frame(frame,keys,y,z,locked,'path').assign(lcc_split=split)
        mm=metrics(y,pad_score(z),locked,len(frame))
        write_json(folder/'metrics'/f'lcc_{split}_metrics.json',mm)
        write_json(folder/'metrics'/f'lcc_{split}_reference_metrics.json',reference_metrics(y,z,len(frame)))
        pred.to_csv(folder/'predictions'/f'lcc_{split}_predictions.csv',index=False)
        scored_predictions.append(pred)
    pooled=pd.concat(scored_predictions,ignore_index=True)
    if not pooled[['path','label','status','usable_for_pad','bbox_origin','lcc_split']].equals(
        lcc_frames['combined'][['path','label','status','usable_for_pad','bbox_origin','lcc_split']]):
        raise RuntimeError('Combined prediction candidates differ from 00d manifest')
    usable=pooled.usable_for_pad.astype(bool)
    mm=metrics(pooled.loc[usable,'label'],pooled.loc[usable,'pad_logit_score'],locked,len(pooled))
    write_json(folder/'metrics'/'lcc_combined_metrics.json',mm)
    write_json(folder/'metrics'/'lcc_combined_reference_metrics.json',reference_metrics(pooled.loc[usable,'label'],pooled.loc[usable,['logit_0','logit_1']].to_numpy(),len(pooled)))
    pooled.to_csv(folder/'predictions'/'lcc_combined_predictions.csv',index=False)

def save_final_source_evaluation(folder,keys,y,z,locked):
    mm=metrics(y,pad_score(z),locked,len(test))
    write_json(folder/'metrics'/'test_metrics.json',mm)
    write_json(folder/'metrics'/'test_reference_metrics.json',reference_metrics(y,z,len(test)))
    prediction_frame(test,keys,y,z,locked,'sample_key').to_csv(folder/'predictions'/'test_predictions.csv',index=False)

def run_one(name,spec):
    folder=out/'runs'/name;folder.mkdir(parents=True,exist_ok=True);(folder/'predictions').mkdir(exist_ok=True);(folder/'metrics').mkdir(exist_ok=True)
    cfg=run_config(name,spec);cfg_path=folder/'config.json'
    if cfg_path.is_file() and read_json(cfg_path)!=cfg:raise RuntimeError(name+' existing config differs')
    write_json(cfg_path,cfg)
    (folder/'manifests').mkdir(exist_ok=True)
    for filename in MANIFEST_SOURCES:shutil.copyfile(out/'manifests'/filename,folder/'manifests'/filename)
    best_path=folder/'best.pth';last_path=folder/'last_state.pth';done_path=folder/'completion.json'
    if done_path.is_file():
        needed=[folder/'metrics'/(k+'_metrics.json') for k in ('val','test','lcc_training','lcc_development','lcc_evaluation','lcc_combined')]
        if not all(p.is_file() for p in needed) or not best_path.is_file() or not (folder/'evaluation_freeze.json').is_file() or not all((folder/'metrics'/(s+'_reference_metrics.json')).is_file() for s in ('test','lcc_training','lcc_development','lcc_evaluation','lcc_combined')):
            raise RuntimeError(name+' incomplete completion outputs')
        print(name,'completed; skipping')
        return
    model,audit=init_model({'init':'pretrained','copied':True})
    if audit['initial_state_sha256']!=INITIAL_STATE_SHA:raise RuntimeError('Initial state differs')
    write_json(folder/'initialization_audit.json',audit)
    model=model.to(device)
    optimizer=torch.optim.SGD((p for p in model.parameters() if p.requires_grad),lr=.005,momentum=.9,weight_decay=5e-4)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=MILESTONES,gamma=GAMMA)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    gain_rng=torch.Generator(device=device).manual_seed(GAIN_SEED)
    best_checkpoint=None
    history=[];diagnostics=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0;no_improve=0;start=1
    if last_path.is_file():
        if not RESUME:raise RuntimeError(name+' has a saved state; enable RESUME or use a new output root')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('run_id')!=name or state.get('config')!=cfg:raise RuntimeError(name+' resume run/config differs')
        if state.get('initial_state_sha256')!=INITIAL_STATE_SHA:raise RuntimeError('Resume initial state differs')
        best_checkpoint=state['best_checkpoint']
        if state['best_epoch']:
            if best_checkpoint['run_id']!=name or best_checkpoint['epoch']!=state['best_epoch'] or best_checkpoint['config']!=cfg:raise RuntimeError('Resume best payload differs')
            save_torch(best_path,best_checkpoint)
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        history=state['history'];diagnostics=state['diagnostics'];best_acer=state['best_acer']
        best_auc=state['best_auc'];best_epoch=state['best_epoch'];no_improve=state['no_improve']
        start=state['epoch']+1
        if len(history)!=state['epoch']:raise RuntimeError(name+' resume history length differs')
        restore_rng(state['rng'],gain_rng)
        print(name,'resume at epoch',start)
    val_dl=loader(CelebaData(val))
    masks=None  # CE-only: no FFT or worst-view selection
    if start<=MAX_EPOCHS and not (history and history[-1]['stop_reached']):
        for epoch in range(start,MAX_EPOCHS+1):
            expected_lr=.005 if epoch<=4 else (.001 if epoch<=8 else .0002)
            if not math.isclose(optimizer.param_groups[0]['lr'],expected_lr,rel_tol=1e-9):raise RuntimeError('Scheduler LR differs')
            warmup=epoch<=spec['warmup_epochs']
            active=not warmup
            model.train();totals=Counter();band_count=Counter();seen=0;harmful_count_total=0;harmful_ce_sum_total=0.;epoch_started=time.monotonic()
            train_ds=CelebaData(train,training=True,copied=True)
            train_dl=loader(train_ds,shuffle=True,epoch=epoch)
            print(f'[{name}] epoch {epoch}/{MAX_EPOCHS} starting, warmup={warmup}',flush=True)
            for step,(x,_,binary,spoof,lighting,attrs,_) in enumerate(train_dl,1):
                x=x.to(device,non_blocking=True);binary=binary.to(device);optimizer.zero_grad(set_to_none=True)
                loss,scalars,bands,harmful_ce_sum=training_terms(model,x,binary,spoof,lighting,attrs,
                    spec,masks,gain_rng,active)
                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
                count=len(x);seen+=count
                for key,value in scalars.items():totals[key]+=value*count
                band_count.update(bands)
                harmful_count_total+=scalars['harmful_count'];harmful_ce_sum_total+=harmful_ce_sum
                if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(train_dl):
                    print(f'[{name}] epoch {epoch} batch {step}/{len(train_dl)}, loss={totals["train_total_loss"]/seen:.4f}, elapsed={time.monotonic()-epoch_started:.1f}s',flush=True)
            keys,y,z=infer(model,val_dl)
            if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val order differs')
            scores=pad_score(z);threshold=threshold_from_val(y,scores)
            vm=metrics(y,scores,threshold,len(test))
            improved=active and (vm['acer']<best_acer-METRIC_TIE_EPS or
                (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if active:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    best_checkpoint={'run_id':name,'epoch':epoch,'config':cfg,
                        'val_acer':best_acer,'val_auc':best_auc,
                        'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}}
                    save_torch(best_path,best_checkpoint)
                no_improve=advance_patience(no_improve,improved,epoch,active)
            row={'epoch':epoch,'phase':'warmup' if warmup else 'active','lr':optimizer.param_groups[0]['lr'],
                 'eligible_for_final_checkpoint':active,'val_acer':vm['acer'],'val_auc':vm['auc'],
                 'val_eer':vm['eer'],'val_threshold':threshold,'best_epoch':best_epoch,
                 'no_improve_count':no_improve,'clean_ce_weight':1.0 if warmup else spec['clean_ce_weight'],
                 'spectral_ce_weight':0.0 if warmup else spec['spectral_ce_weight'],
                 'method':'CLEAN_WARMUP' if warmup else spec['method'],
                 **{k:v/seen for k,v in totals.items()},
                 **{f'{label}_count':band_count[i] for i,label in enumerate(('low','mid','high'))},
                 **{f'{label}_fraction':band_count[i]/seen for i,label in enumerate(('low','mid','high'))}}
            row['harmful_count']=harmful_count_total
            if spec['method']=='HARMFUL':
                row['harmful_ce_batch_mean']=row['harmful_ce']
                row['harmful_ce']=harmful_ce_sum_total/harmful_count_total if harmful_count_total else 0.
            if active and spec['method']!='CLEAN':diagnostics.append(dict(row))
            stop=epoch==MAX_EPOCHS or patience_stop_reached(epoch,no_improve)
            row['stop_reached']=stop;history.append(row)
            scheduler.step()
            state={'run_id':name,'config':cfg,'epoch':epoch,
                'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'no_improve':no_improve,'initial_state_sha256':INITIAL_STATE_SHA,'rng':snapshot_rng(gain_rng),'best_checkpoint':best_checkpoint,
                'history':history,'diagnostics':diagnostics}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            if spec['method']!='CLEAN':pd.DataFrame(diagnostics).to_csv(folder/'spectral_diagnostics.csv',index=False)
            print(name,'epoch',epoch,'Full Test-as-Val ACER',vm['acer'],'best',best_epoch,'patience',no_improve,flush=True)
            if stop:break
    if not best_path.is_file() or best_epoch<=spec['warmup_epochs']:
        raise RuntimeError(name+' has no eligible best checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=True)
    if best['run_id']!=name or best['config']!=cfg or best['epoch']!=best_epoch:
        raise RuntimeError(name+' best checkpoint identity differs')
    model.load_state_dict(best['state_dict']);model.eval()
    keys,y,z=infer(model,val_dl)
    if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Frozen Val order differs')
    locked=threshold_from_val(y,pad_score(z))
    vm=metrics(y,pad_score(z),locked,len(test))
    if abs(vm['acer']-best['val_acer'])>1e-6 or abs(vm['auc']-best['val_auc'])>1e-6:
        raise RuntimeError('Best checkpoint does not reproduce Val selection metrics')
    write_json(folder/'metrics'/'val_metrics.json',vm)
    save_final_source_evaluation(folder,keys,y,z,locked)
    freeze={'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,'selection_set':'Full official Test-as-Val','checkpoint_sha256':sha(best_path),'source_metrics':vm,'val_metrics':vm,'manifest_sha256':MANIFEST_HASHES,
      'train_crop_factor':1.5,'val_crop_factor':1.5,'test_crop_factor':1.5,'lcc_crop_factor':1.5}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.is_file() and read_json(freeze_path)!=freeze:raise RuntimeError('Evaluation freeze differs')
    write_json(freeze_path,freeze)
    evaluate_lcc(model,folder,locked)
    write_json(done_path,{'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,
                          'initial_state_sha256':INITIAL_STATE_SHA})
    del model
    torch.cuda.empty_cache()

def assert_fairness_contract(configs,initial_hashes):
    if set(configs)!=set(RUNS) or set(initial_hashes)!=set(RUNS):raise RuntimeError('Baseline run identity mismatch')
    for name,cfg in configs.items():
        if cfg['manifest_sha256']!=MANIFEST_HASHES or cfg['initial_state_sha256']!=initial_hashes[name]:raise RuntimeError('Manifest/initial state changed')
        if cfg['loss']!={'CE_clean':1.,'CE_worst':0.,'L_aux':0.} or cfg['spectral'] is not None:raise RuntimeError('CE-only objective changed')
        if any(cfg[k]!=1.5 for k in ('train_crop_factor','val_crop_factor','test_crop_factor','lcc_crop_factor')):raise RuntimeError('Crop mismatch')
    return True


def advance_patience(patience_counter,improved,epoch,eligible):
    # Checkpoint eligibility and improvement are determined before this rule.
    if improved or epoch<PATIENCE_START_EPOCH:return 0
    if not eligible:return patience_counter
    return patience_counter+1

def patience_stop_reached(epoch,patience_counter):
    return epoch>=PATIENCE_START_EPOCH and patience_counter>=EARLY_STOP_PATIENCE

def phase_patience_sanity_checks():
    for spec in RUNS.values():
        counter=0;stopped=None
        for epoch in range(1,MAX_EPOCHS+1):
            eligible=epoch>spec['warmup_epochs']
            counter=advance_patience(counter,False,epoch,eligible)
            if epoch<9:assert counter==0
            if patience_stop_reached(epoch,counter):stopped=epoch;break
        assert stopped==11
    counter=advance_patience(2,True,9,True)
    assert counter==0
    for epoch in (10,11,12):counter=advance_patience(counter,False,epoch,True)
    assert counter==3 and patience_stop_reached(12,counter)
    assert advance_patience(0,True,5,True)==0
    assert not patience_stop_reached(8,3)
    # Resuming committed epoch8 vs uninterrupted final-phase counts gives identical results.
    first=advance_patience(0,False,9,True);resumed=advance_patience(first,False,10,True)
    assert resumed==2 and advance_patience(resumed,False,11,True)==3
    return {'patience_start_epoch':9,'early_stop_patience':3,'earliest_possible_stop_epoch':11,
            'all_methods_identical':True,'improvement_resets_counter':True,'epoch_boundary_counter_replay':True,
            'best_selection_not_delayed':True}


def reference_metrics(y3,logits,candidate_n):
    y=(np.asarray(y3,dtype=int)!=0).astype(int)  # Reference 0=real, 1=attack.
    z=torch.as_tensor(np.asarray(logits));p=F.softmax(z,dim=-1)[:,1].cpu().numpy();pred=z.argmax(dim=1).cpu().numpy()
    real=y==0;attack=y==1
    if not real.any() or not attack.any():raise ValueError('Reference metrics need both classes')
    fpr,tpr,_=roc_curve(y,p,pos_label=1);idx=int(np.nanargmin(np.abs((1-tpr)-fpr)))
    apcer=float((pred[real]==1).mean());bpcer=float((pred[attack]==0).mean())
    return {'auc':float(roc_auc_score(y,p)),'eer':float(min(fpr[idx],1-tpr[idx])),
      'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),'global_argmax_accuracy':float((pred==y).mean()),
      'candidate_n':int(candidate_n),'scored_n':len(y),'coverage':len(y)/candidate_n,
      'convention':'Reference variable names: APCER=real->attack; BPCER=attack->real; argmax logits; ROC attack probability'}


GPU: Tesla T4 Full Train: 484561 Full Test-as-Val usable/candidate: 66379 67170 LCC: {'training': 8299, 'development': 2948, 'evaluation': 7580, 'combined': 18827}


## CE-only objective and training controls

Auxiliary modules retain historical initialization/state-dict compatibility, but are frozen and never forwarded. One clean forward per training batch. No warmup; checkpoint eligibility from epoch1.


In [4]:
# Lightweight objective/gradient checks before training. No dataset/model training.
assert_fairness_contract({name:run_config(name,spec) for name,spec in RUNS.items()},INITIAL_HASHES)
write_json(out/'phase_patience_checks.json',phase_patience_sanity_checks())
def check_ce_only_objective():
    class TinyPAD(nn.Module):
        def __init__(self):
            super().__init__();self.head=nn.Linear(3,2);self.calls=0
        def forward(self,x,aux=False):
            assert not aux,'Auxiliary forward must not be requested'
            self.calls+=1;return self.head(x)
    global use_amp
    old=use_amp;use_amp=False
    try:
        model=TinyPAD();x=torch.tensor([[.1,.2,.3],[.6,.4,.2]]);y=torch.tensor([0,1])
        expected=F.cross_entropy(model(x),y);expected_grad=torch.autograd.grad(expected,model.head.weight)[0]
        model.calls=0
        loss,diagnostics,_,_=training_terms(model,x,y,None,None,None,RUNS['CE_CLEAN_FULL_crop15'],None,None,True)
        actual_grad=torch.autograd.grad(loss,model.head.weight)[0]
        assert model.calls==1 and torch.allclose(actual_grad,expected_grad)
        assert diagnostics['aux_loss']==diagnostics['spectral_ce']==0.
    finally:use_amp=old
    return {'single_clean_forward':True,'gradient_equals_binary_ce':True,'auxiliary_loss':0.,'spectral_loss':0.}
write_json(out/'ce_only_checks.json',check_ce_only_objective())
print('CE-only objective and phase-aware patience checks passed',flush=True)


CE-only objective and phase-aware patience checks passed


/tmp/ipykernel_23/1895080082.py:312: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


## CASIA evaluation

This function embeds notebook20 preprocessing, coverage, aggregation and train-only calibration. It uses the new best checkpoint and isolates target manifests from source training state.


In [5]:
def evaluate_casia(model,folder,name,locked,best_epoch):
    OUTPUT_ROOT=folder/'casia';OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    CASIA=Path(CASIA_ROOT);detector_path=Path(SCRFD_MODEL_PATH)
    SELECTED_RUNS=[name];SEED=TRAIN_SEED
    LABELS={name:'CE-clean only (full-scale)'}
    casia_info={'run_id':name,'scale':'full-scale','best_epoch':best_epoch,'threshold':locked,
                'amp':use_amp,'loss':'CE_clean only','crop_factor':1.5}
    provenance=[casia_info]
    # Full train/test-color manifests, validated explicit filename labels; no frame subsampling.
    EXPECTED_CODES={'1':(0,'real','low'),'2':(0,'real','normal'),'HR_1':(0,'real','high'),
     '3':(1,'warped','low'),'4':(1,'warped','normal'),'HR_2':(1,'warped','high'),
     '5':(1,'cut','low'),'6':(1,'cut','normal'),'HR_3':(1,'cut','high'),
     '7':(1,'replay','low'),'8':(1,'replay','normal'),'HR_4':(1,'replay','high')}
    FILENAME_RE=re.compile(r'^(?P<subject>\d+)_(?P<code>HR_[1-4]|[1-8])\.avi_(?P<frame>\d+)_(?P<label>real|fake)\.jpg$',re.I)
    def read_split(folder):
        output=[]
        for path in sorted(folder.iterdir()):
            if path.name.startswith('.'):continue
            match=FILENAME_RE.fullmatch(path.name)
            if not path.is_file() or not match:raise ValueError('Unexpected CASIA filename: '+str(path))
            g=match.groupdict();code=g['code'].upper();label,attack,quality=EXPECTED_CODES[code]
            if g['label'].lower()!=('real' if label==0 else 'fake'):raise ValueError('Filename label/code conflict: '+str(path))
            output.append({'subject_id':int(g['subject']),'video_code':code,'video_id':f'{int(g["subject"])}_{code}.avi',
                           'frame_index':int(g['frame']),'image_path':str(path),'binary_label':label,
                           'attack_type':attack,'quality':quality})
        frame=pd.DataFrame(output)
        if frame.empty or frame.duplicated(['video_id','frame_index']).any():raise ValueError('Empty/duplicate manifest: '+str(folder))
        return frame.sort_values(['subject_id','video_id','frame_index']).reset_index(drop=True)
    # ID collision checks use explicit labels, never PAD scores or detection results.
    train_names=read_split(CASIA/'train_img/train_img/color').assign(split='train')
    test_names=read_split(CASIA/'test_img/test_img/color').assign(split='test')
    frame_manifest=pd.concat([train_names,test_names],ignore_index=True)
    for vid,group in frame_manifest.groupby('video_id'):
        if len(group[['subject_id','video_code','binary_label','attack_type','quality']].drop_duplicates())!=1:
            raise ValueError('Inconsistent video labels across splits: '+vid)
    def combined_unique(frame):
        # Stable train then test ordering; test takes precedence independently of detection success.
        ordered=pd.concat([frame.loc[frame.split=='train'],frame.loc[frame.split=='test']],ignore_index=True)
        result=ordered.drop_duplicates(['video_id','frame_index'],keep='last').copy()
        result['source_split']=result['split']
        result['split']='combined'
        return result.sort_values(['subject_id','video_id','frame_index']).reset_index(drop=True)
    combined_manifest=combined_unique(frame_manifest)
    POOLS=['train','test','combined']
    pool_manifests={'train':train_names,'test':test_names,'combined':combined_manifest}
    def video_manifest(frame):
        return frame.groupby(['split','video_id'],sort=True).agg(subject_id=('subject_id','first'),
          video_code=('video_code','first'),binary_label=('binary_label','first'),
          attack_type=('attack_type','first'),quality=('quality','first'),candidate_frames=('frame_index','size')).reset_index()
    videos=pd.concat([video_manifest(pool_manifests[split]) for split in POOLS],ignore_index=True)
    for split,frame in pool_manifests.items():
        if set(frame.binary_label)!={0,1}:raise ValueError('Real and Attack required in '+split)
        frame.to_csv(OUTPUT_ROOT/f'casia_{split}_frame_manifest.csv',index=False)
    frame_manifest.to_csv(OUTPUT_ROOT/'casia_frame_manifest.csv',index=False)
    videos.to_csv(OUTPUT_ROOT/'casia_video_manifest.csv',index=False)
    overlap={'subjects':sorted(set(test_names.subject_id)&set(train_names.subject_id)),
             'video_ids':len(set(test_names.video_id)&set(train_names.video_id)),
             'frame_ids':len(set(zip(test_names.video_id,test_names.frame_index))&set(zip(train_names.video_id,train_names.frame_index)))}
    populations={split:{'candidate_frames':len(pool_manifests[split]),
                        'candidate_videos':int((videos.split==split).sum())} for split in POOLS}
    protocol={'dataset':'CASIA-FASD Kaggle extracted-image train + test evaluation (minhnh2107/casiafasd)',
     'official_subject_disjoint_protocol_verified':False,'splits':['train_img/train_img/color','test_img/test_img/color'],
     'train_test_copy_overlap':overlap,'frame_sampling':'none; all available train and test color JPEGs',
     'raw_train_test_frames':len(frame_manifest),'populations':populations,
     'combined_policy':'unique (video_id,frame_index); prefer test copy before detection; no duplicate weighting','selected_runs':SELECTED_RUNS,
     'detector':'SCRFD-500M','detector_path':str(detector_path),'detector_sizes':DETECTOR_SIZES,
     'detector_confidence':DETECTOR_CONFIDENCE,'face_selection':'largest positive finite bbox; confidence tie-break',
     'crop_factor':CROP_SCALE,'input_size':INPUT_SIZE,'color_order':'BGR','input_range':'[0,1]',
     'score':'logit(real)-logit(attack)','primary_unit':'reconstructed video; mean valid-frame score',
     'min_valid_frames':MIN_VALID_FRAMES,'secondary_unit':'all successfully localized frames in the specified pool',
     'threshold_policy':'source-frozen baseline plus CASIA train-calibrated frame/video thresholds' if CALIBRATE_CASIA_THRESHOLD else 'source-frozen only',
     'eer_policy':'descriptive ROC EER only; its target threshold is never used for predictions',
     'target_training':False,'target_adaptation':False,'target_threshold_calibration':CALIBRATE_CASIA_THRESHOLD,
     'threshold_fit_split':'train only' if CALIBRATE_CASIA_THRESHOLD else None,
     'threshold_objective':'minimum pooled binary ACER/HTER; ties closest to source threshold',
     'full_scale_source_selection':'CelebA Test-as-Val, as in experiment18',
     'inference_precision':'per-run original AMP setting; logits converted to float32 before subtraction',
     'seed':SEED,'batch_size':BATCH_SIZE,'torch_version':torch.__version__,'ort_version':ort.__version__}
    write_json(OUTPUT_ROOT/'evaluation_protocol.json',protocol)
    print('Candidate populations:',populations,flush=True)
    print('Train/test image-copy overlap:',overlap,flush=True)
    display(videos.groupby(['split','binary_label','attack_type','quality']).size().rename('videos').reset_index())

    # Localize once and cache crops in RAM. Every selected model receives identical tensors.
    # Repeat this cell only to rebuild detection; no target bbox/GT fallback is used.
    detector=model_zoo.get_model(str(detector_path),providers=['CUDAExecutionProvider','CPUExecutionProvider'])
    if detector is None:raise RuntimeError('Could not load SCRFD-500M')
    detector.prepare(ctx_id=0,input_size=DETECTOR_SIZES[0],det_thresh=DETECTOR_CONFIDENCE)
    active=detector.session.get_providers()
    # CPU provider may handle shape operations; CUDA must actually be active.
    if 'CUDAExecutionProvider' not in active:raise RuntimeError('SCRFD CUDA failed to load; active providers: '+str(active))
    print('SCRFD active providers:',active,flush=True)
    def detect_boxes(image,size):
        try:return detector.detect(image,input_size=size,max_num=0,metric='default')[0]
        except TypeError:
            previous=detector.input_size;detector.input_size=size
            try:return detector.detect(image,max_num=0,metric='default')[0]
            finally:detector.input_size=previous
    patches=[];records=[];started=time.monotonic()
    print('[Detection] Starting shared full train/test localization',flush=True)
    for i,row in enumerate(frame_manifest.to_dict('records')):
        rec=dict(row,decoded=False,detected=False,crop_index=-1,bbox_xyxy='',confidence=None,detector_size=None,failure_reason='')
        image=cv2.imread(row['image_path'],cv2.IMREAD_COLOR)
        if image is None:rec['failure_reason']='decode_failed'
        else:
            rec['decoded']=True
            try:
                for size in DETECTOR_SIZES:
                    boxes=detect_boxes(image,size)
                    if boxes is None:continue
                    boxes=[b for b in boxes if len(b)>=5 and np.isfinite(b[:5]).all() and b[2]>b[0] and b[3]>b[1] and b[4]>=DETECTOR_CONFIDENCE]
                    if not boxes:continue
                    box=max(boxes,key=lambda b:((float(b[2])-float(b[0]))*(float(b[3])-float(b[1])),float(b[4])))
                    patch=mini_crop_bgr(image,box[:4])
                    rec.update(detected=True,crop_index=len(patches),bbox_xyxy=json.dumps([float(v) for v in box[:4]]),
                               confidence=float(box[4]),detector_size=size[0])
                    patches.append(patch);break
                if not rec['detected']:rec['failure_reason']='scrfd_no_face'
            except Exception as exc:rec['failure_reason']=f'{type(exc).__name__}: {exc}'
        records.append(rec)
        if i==0 or (i+1)%DETECTION_LOG_EVERY==0 or i+1==len(frame_manifest):
            print(f'[Detection] {i+1}/{len(frame_manifest)} frames; valid={len(patches)}; elapsed={time.monotonic()-started:.1f}s',flush=True)
        if (i+1)%DETECTION_LOG_EVERY==0:pd.DataFrame(records).to_csv(OUTPUT_ROOT/'casia_frame_detection.partial.csv',index=False)
    detection=pd.DataFrame(records);detection.to_csv(OUTPUT_ROOT/'casia_frame_detection.csv',index=False)
    if not patches:raise RuntimeError('No valid face crops; inspect casia_frame_detection.csv')
    CROPS=np.stack(patches);del patches
    valid=detection.loc[detection.detected].copy().reset_index(drop=True)
    assert valid.crop_index.tolist()==list(range(len(CROPS)))
    np.save(OUTPUT_ROOT/'shared_crops_bgr_uint8.npy',CROPS)
    # Pooled rows are fixed by the pre-detection manifest policy, including failed test copies.
    pool_detection={'train':detection.loc[detection.split=='train'].copy(),
                    'test':detection.loc[detection.split=='test'].copy(),
                    'combined':combined_unique(detection)}
    combined_rows=pool_detection['combined']
    assert combined_rows.image_path.tolist()==combined_manifest.image_path.tolist()
    combined_rows.to_csv(OUTPUT_ROOT/'casia_combined_frame_detection.csv',index=False)
    coverages=[]
    for split in POOLS:
        frame=pool_detection[split]
        cov=frame.groupby(['split','video_id'],sort=True).agg(decoded_frames=('decoded','sum'),valid_frames=('detected','sum')).reset_index()
        cov=videos.loc[videos.split==split].merge(cov,on=['split','video_id'],validate='one_to_one')
        cov['scorable']=cov.valid_frames>=MIN_VALID_FRAMES
        cov['detection_coverage']=cov.valid_frames/cov.candidate_frames
        coverages.append(cov)
    coverage=pd.concat(coverages,ignore_index=True)
    coverage.to_csv(OUTPUT_ROOT/'casia_video_coverage.csv',index=False)
    valid.groupby(['split','binary_label','attack_type','quality']).size().rename('valid_frames').reset_index().to_csv(OUTPUT_ROOT/'valid_frames_by_group.csv',index=False)
    protocol['detector_active_providers']=active
    write_json(OUTPUT_ROOT/'evaluation_protocol.json',protocol)
    print(f'[Detection complete] {len(valid)}/{len(frame_manifest)} raw train/test frames',flush=True)
    display(coverage.groupby('split').agg(candidate_videos=('video_id','size'),scorable_videos=('scorable','sum'),candidate_frames=('candidate_frames','sum'),valid_frames=('valid_frames','sum')).reset_index())
    display(detection.failure_reason.value_counts())
    del detector;gc.collect();torch.cuda.empty_cache()

    # Frozen checkpoint inference, one model at a time. Progress printed immediately and per batch.
    def safe_metrics(labels,scores,threshold,candidate_n):
        if len(labels)==0 or set(np.asarray(labels)==0)!={False,True}:
            y=np.asarray(labels);pred=np.asarray(scores)>=threshold;real=y==0;attack=y!=0
            return dict(auc=None,eer=None,tpr_at_fpr_1pct=None,
                        apcer=float(pred[attack].mean()) if attack.any() else None,
                        bpcer=float((~pred[real]).mean()) if real.any() else None,acer=None,hter=None,
                        binary_accuracy=float((pred==real).mean()) if len(y) else None,
                        locked_source_threshold=float(threshold),candidate_n=int(candidate_n),
                        scored_n=len(labels),detector_coverage=len(labels)/candidate_n if candidate_n else 0)
        return metrics(labels,scores,threshold,candidate_n)
    summary=[];all_frame_predictions=[];all_video_predictions=[];subgroup_rows=[]
    for run_id in SELECTED_RUNS:
        info=casia_info;model=model.to(device);threshold=info['threshold']
        run_output=OUTPUT_ROOT;run_output.mkdir(parents=True,exist_ok=True)
        started=time.monotonic();chunks=[];num_batches=math.ceil(len(CROPS)/BATCH_SIZE)
        print(f'\n[Inference] {run_id}: {len(CROPS)} shared crops, {num_batches} batches; threshold={threshold}',flush=True)
        with torch.inference_mode():
            for batch_no,start in enumerate(range(0,len(CROPS),BATCH_SIZE),1):
                arr=np.ascontiguousarray(CROPS[start:start+BATCH_SIZE].transpose(0,3,1,2))
                x=torch.from_numpy(arr).to(device=device,dtype=torch.float32).div_(255.)
                with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=info['amp']):z=model(x)
                z=z.float().cpu().numpy()
                if z.shape!=(len(arr),2) or not np.isfinite(z).all():raise ValueError('Nonfinite / nonbinary model logits: '+run_id)
                chunks.append(z)
                if batch_no==1 or batch_no%LOG_EVERY_BATCHES==0 or batch_no==num_batches:
                    print(f'[Inference] {run_id} batch {batch_no}/{num_batches}; elapsed={time.monotonic()-started:.1f}s',flush=True)
        logits=np.concatenate(chunks)
        fp=valid[['split','subject_id','video_id','video_code','frame_index','image_path','binary_label','attack_type','quality']].copy()
        fp['run_id']=run_id;fp['logit_real']=logits[:,0];fp['logit_attack']=logits[:,1]
        fp['pad_score']=logits[:,0]-logits[:,1];fp['threshold']=threshold
        fp['predicted_real']=fp.pad_score>=threshold
        fp.to_csv(run_output/'frame_predictions.csv',index=False)
        all_frame_predictions.append(fp)
        write_json(run_output/'config.json',info|{'training':False,'pools':POOLS})
        for split in POOLS:
            # Reindex predictions to candidate rows chosen BEFORE detection. No fallback to a train
            # duplicate when its preferred test copy failed; this prevents outcome-based selection.
            candidates=pool_detection[split]
            chosen=candidates.loc[candidates.detected].copy()
            columns=['image_path','logit_real','logit_attack','pad_score','threshold','predicted_real']
            sf=chosen.merge(fp[columns],on='image_path',how='left',validate='one_to_one')
            if sf.pad_score.isna().any():raise ValueError('Missing predictions in '+split)
            means=sf.groupby('video_id',sort=True).agg(video_score=('pad_score','mean')).reset_index()
            cv=coverage.loc[coverage.split==split].copy()
            vp=cv.merge(means,on='video_id',how='left',validate='one_to_one')
            vp['run_id']=run_id;vp['threshold']=threshold
            vp['predicted_real']=pd.Series(pd.NA,index=vp.index,dtype='boolean')
            vp.loc[vp.scorable,'predicted_real']=(vp.loc[vp.scorable,'video_score']>=threshold).to_numpy()
            scored=vp.loc[vp.scorable]
            video_metrics=safe_metrics(scored.binary_label.to_numpy(),scored.video_score.to_numpy(),threshold,len(cv))
            frame_metrics=safe_metrics(sf.binary_label.to_numpy(),sf.pad_score.to_numpy(),threshold,len(candidates))
            split_output=run_output/split;split_output.mkdir(exist_ok=True)
            sf.to_csv(split_output/'frame_predictions.csv',index=False);vp.to_csv(split_output/'video_predictions.csv',index=False)
            write_json(split_output/'video_metrics.json',video_metrics);write_json(split_output/'frame_metrics.json',frame_metrics)
            summary.append(info|{'split':split}|video_metrics|{'frame_'+k:v for k,v in frame_metrics.items()})
            all_video_predictions.append(vp)
            for group_col in ['attack_type','quality']:
                for group,df in scored.groupby(group_col):
                    subgroup_rows.append({'run_id':run_id,'split':split,'group_by':group_col,'group':group,
                      **safe_metrics(df.binary_label.to_numpy(),df.video_score.to_numpy(),threshold,int((cv[group_col]==group).sum()))})
            print(f'[Result] {run_id} / {split}: video AUC={video_metrics["auc"]}, HTER={video_metrics["hter"]}, frame AUC={frame_metrics["auc"]}',flush=True)
        # Save summaries after every completed run so interruption preserves completed results.
        pd.DataFrame(summary).to_csv(OUTPUT_ROOT/'comparison_summary.csv',index=False)
        write_json(OUTPUT_ROOT/'comparison_summary.json',summary)
        write_json(run_output/'completion.json',{'run_id':run_id,'status':'completed','elapsed_seconds':time.monotonic()-started})
        print(f'[Done] {run_id}: train / test / combined saved',flush=True)
        del model,logits,chunks,x,z;gc.collect();torch.cuda.empty_cache()
    frame_predictions=pd.concat(all_frame_predictions,ignore_index=True)
    video_predictions=pd.concat(all_video_predictions,ignore_index=True)
    # All model comparisons are paired on exactly the same ordered localized frames/videos.
    keys=[df.image_path.tolist() for df in all_frame_predictions]
    assert all(k==keys[0] for k in keys)
    frame_predictions.to_csv(OUTPUT_ROOT/'all_frame_predictions.csv',index=False)
    video_predictions.to_csv(OUTPUT_ROOT/'all_video_predictions.csv',index=False)
    pd.DataFrame(subgroup_rows).to_csv(OUTPUT_ROOT/'video_subgroup_metrics.csv',index=False)
    comparison=pd.DataFrame(summary)
    display(comparison[['run_id','split','best_epoch','locked_source_threshold','auc','eer','apcer','bpcer','acer','hter','binary_accuracy','tpr_at_fpr_1pct','scored_n','candidate_n','frame_auc','frame_acer']])

    def optimize_casia_threshold(labels, scores, source_threshold):
        y = np.asarray(labels, dtype=int)
        s = np.asarray(scores, dtype=float)
        if y.ndim != 1 or s.ndim != 1 or len(y) != len(s):
            raise ValueError('Labels and scores must be matching one-dimensional arrays')
        if not np.isfinite(s).all() or not np.isfinite(source_threshold) or not set(y).issubset({0, 1}):
            raise ValueError('Invalid CASIA scores, labels or source threshold')
        if not len(y) or set(y) != {0, 1}:
            return None, pd.DataFrame()
        # Every distinct >= decision is considered, including accepting nobody.
        thresholds = np.unique(np.r_[s, np.nextafter(s.max(), np.inf), float(source_threshold)])
        real_scores = np.sort(s[y == 0])
        attack_scores = np.sort(s[y == 1])
        bpcer = np.searchsorted(real_scores, thresholds, side='left') / len(real_scores)
        apcer = 1 - np.searchsorted(attack_scores, thresholds, side='left') / len(attack_scores)
        hter = (apcer + bpcer) / 2
        candidates = np.flatnonzero(np.isclose(hter, hter.min(), rtol=0, atol=1e-12))
        best = min(candidates, key=lambda i: (abs(thresholds[i] - source_threshold), thresholds[i]))
        curve = pd.DataFrame({'threshold': thresholds, 'apcer': apcer, 'bpcer': bpcer,
                              'acer': hter, 'hter': hter})
        return float(thresholds[best]), curve

    # Reuse predictions; never repeat model inference or fit on test/combined.
    calibrated_rows = []
    if CALIBRATE_CASIA_THRESHOLD:
        for run_id in SELECTED_RUNS:
            run_output = OUTPUT_ROOT
            source_threshold = float(next(r for r in summary if r['run_id'] == run_id)['locked_source_threshold'])
            for unit, score_column in [('frame', 'pad_score'), ('video', 'video_score')]:
                train = pd.read_csv(run_output / 'train' / f'{unit}_predictions.csv')
                if unit == 'video':
                    train = train.loc[train.scorable]
                threshold, curve = optimize_casia_threshold(train.binary_label.to_numpy(), train[score_column].to_numpy(), source_threshold)
                calibration = {'run_id': run_id, 'unit': unit, 'fit_split': 'train',
                               'fit_n': len(train), 'source_threshold': source_threshold,
                               'casia_calibrated_threshold': threshold,
                               'objective': 'minimum pooled binary ACER/HTER',
                               'tie_break': 'closest to source threshold',
                               'train_test_copy_overlap': overlap,
                               'status': 'calibrated' if threshold is not None else 'undefined: both train classes required'}
                write_json(run_output / f'{unit}_casia_calibration.json', calibration)
                if threshold is not None:
                    curve.to_csv(run_output / f'{unit}_casia_train_threshold_curve.csv', index=False)
                for split in POOLS:
                    split_output = run_output / split
                    predictions = pd.read_csv(split_output / f'{unit}_predictions.csv')
                    eligible = predictions if unit == 'frame' else predictions.loc[predictions.scorable]
                    candidate_n = populations[split]['candidate_frames' if unit == 'frame' else 'candidate_videos']
                    baseline = safe_metrics(eligible.binary_label.to_numpy(), eligible[score_column].to_numpy(), source_threshold, candidate_n)
                    row = calibration | {'split': split, 'source_acer': baseline['acer'], 'source_hter': baseline['hter'],
                          'evaluation_policy': 'CASIA train threshold transferred unchanged; no test/combined fitting'}
                    if threshold is not None:
                        tuned = safe_metrics(eligible.binary_label.to_numpy(), eligible[score_column].to_numpy(), threshold, candidate_n)
                        tuned.pop('locked_source_threshold')
                        row.update(tuned)
                        row['acer_reduction'] = baseline['acer'] - tuned['acer'] if tuned['acer'] is not None else None
                        row['hter_reduction'] = row['acer_reduction']
                        predictions['source_threshold'] = source_threshold
                        predictions['threshold'] = threshold
                        predictions['threshold_fit_split'] = 'train'
                        predictions['predicted_real'] = pd.Series(pd.NA, index=predictions.index, dtype='boolean')
                        mask = predictions[score_column].notna()
                        if unit == 'video': mask &= predictions.scorable
                        predictions.loc[mask, 'predicted_real'] = (predictions.loc[mask, score_column] >= threshold).to_numpy()
                        predictions.to_csv(split_output / f'{unit}_predictions_casia_calibrated.csv', index=False)
                    else:
                        row.update({'acer': None, 'hter': None, 'apcer': None, 'bpcer': None, 'acer_reduction': None, 'hter_reduction': None})
                    write_json(split_output / f'{unit}_metrics_casia_calibrated.json', row)
                    calibrated_rows.append(row)
        calibrated_comparison = pd.DataFrame(calibrated_rows)
        calibrated_comparison.to_csv(OUTPUT_ROOT / 'casia_calibrated_threshold_summary.csv', index=False)
        write_json(OUTPUT_ROOT / 'casia_calibrated_threshold_summary.json', calibrated_rows)
        print('CASIA train-calibrated thresholds transferred unchanged to test/combined:', flush=True)
        display(calibrated_comparison[['run_id', 'split', 'unit', 'source_threshold', 'casia_calibrated_threshold',
                                     'source_acer', 'acer', 'source_hter', 'hter', 'apcer', 'bpcer']])
    else:
        print('CASIA calibration disabled; source-frozen results only.', flush=True)

    # Report, ROC figures, and downloadable ZIP. Source-frozen baseline and optional train-calibrated operating points.
    for scale in ['100K','full-scale']:
        ids=[info['run_id'] for info in provenance if info['scale']==scale]
        if not ids:continue
        for split in POOLS:
            fig,ax=plt.subplots(figsize=(7,6))
            for run_id in ids:
                df=video_predictions.loc[(video_predictions.run_id==run_id)&(video_predictions.split==split)&video_predictions.scorable]
                if set(df.binary_label)!={0,1}:continue
                fpr,tpr,_=roc_curve((df.binary_label==0).astype(int),df.video_score)
                ax.plot(fpr,tpr,label=LABELS[run_id])
            ax.plot([0,1],[0,1],'k--',alpha=.5)
            ax.set(xlabel='Attack acceptance rate (FPR)',ylabel='Real acceptance rate (TPR)',title=f'CASIA Kaggle image copy — {scale} / {split}, video ROC')
            handles,_=ax.get_legend_handles_labels()
            if handles:ax.legend()
            fig.tight_layout();fig.savefig(OUTPUT_ROOT/f'roc_video_{scale}_{split}.png',dpi=200);plt.close(fig)
    lines=['# CASIA-FASD Kaggle train/test image-copy evaluation','',
     'No CASIA weight training, adaptation or checkpoint selection. Optional thresholds are fitted only on train, then transferred unchanged to test and combined.', '',
     'All available train and test color JPEGs were attempted; detection/crop failures remain in coverage CSVs. No GT/full-image bbox fallback.',
     'The six checkpoints share the same ordered SCRFD detections, crop1.5 and 80×80 BGR tensors. Spectral views and auxiliary heads are training-only.', '',
     '## Dataset and metric units','',
     f'Raw train/test candidate frames: {len(frame_manifest)}; valid raw frames: {len(valid)}. Populations: {populations}.',
     'Train and test are evaluated separately. Combined is the unique union of reconstructed frame IDs, preferring test copies before detection; it does not count overlapping frame IDs twice.',
     'Combined merges unique frames of the same reconstructed video and averages their valid-frame scores. The two split-wise video rows are not averaged or counted twice.',
     f'Primary metrics: mean valid-frame binary-logit score per reconstructed video; at least {MIN_VALID_FRAMES} valid frames. Secondary metrics: all valid frames.',
     'The frozen source threshold is transferred unchanged to both units; video aggregation may change score distributions. Optional frame/video thresholds are calibrated separately on train.',
     'AUC/EER/TPR@FPR1% describe the ROC. Baseline APCER/BPCER/ACER/HTER/accuracy use the frozen source threshold. EER does not set the prediction threshold.',
     'Metrics are fractions (multiply by 100 for percentages); ACER = HTER = (APCER+BPCER)/2 for the pooled binary attack class.',
     'AUC/EER/ACER/HTER need both classes; single-class subsets still report their applicable APCER or BPCER and accuracy. Undefined metrics remain null.', '',
     '## Limitations','',
     'These are extracted-image Kaggle train and test splits, not a verified official subject-disjoint CASIA benchmark. No original full-video evaluation is claimed.',
     'Filename real/fake labels are cross-checked against video-code mapping from the prior CASIA exploration. No label is guessed from a split folder.',
     f'Train/test image-copy overlap audit: {overlap}. Models were trained on CelebA, not on the CASIA train images.',
     'Face selection is heuristic: largest positive finite SCRFD box, confidence tie-break. Raw bbox records and failures are exported.',
     '100K source selection/calibration: Val15K. Full-scale source selection/calibration: official CelebA Test-as-Val (reused source selection set). Scale groups remain separate.',
     'No statistical significance or best-model selection on CASIA is claimed.', '',
     '## Source-frozen baseline: selected runs and metrics','']
    columns=['run_id','split','auc','eer','apcer','bpcer','acer','hter','binary_accuracy','scored_n','candidate_n','frame_auc','frame_acer']
    lines += ['| '+' | '.join(columns)+' |','| '+' | '.join(['---']*len(columns))+' |']
    for row in summary:
        values=[str(row[c]) if row[c] is not None else 'undefined' for c in columns]
        lines.append('| '+' | '.join(values)+' |')
    lines += ['', 'Source provenance and thresholds: `model_provenance.json`; effective settings: `evaluation_protocol.json`.',
              'Frame/video predictions and metrics: `<train|test|combined>/`.',
              'Use comparison_summary.csv/json to merge independent RUN_FILTER notebooks. Check evaluation protocol and coverage match before merging.','']
    if CALIBRATE_CASIA_THRESHOLD:
        lines += ['', '## CASIA train-calibrated thresholds', '',
                  'Separate frame/video thresholds minimize ACER = HTER on train scores only; ties prefer the source threshold. Real is accepted iff score >= threshold.',
                  'Exactly the same fitted threshold is applied to train, test and combined. Test/combined labels are never used to choose it; no extra inference is performed.',
                  'Train metrics are fitted-set diagnostics; combined includes calibration data. Train/test identity overlap means this Kaggle test is not confirmed independent validation.',
                  'AUC/EER are unchanged. Lower test/combined ACER or HTER is not guaranteed. These thresholds require held-out webcam validation before deployment.', '']
        calibrated_columns = ['run_id', 'split', 'unit', 'source_threshold', 'casia_calibrated_threshold',
                              'source_acer', 'acer', 'source_hter', 'hter', 'apcer', 'bpcer']
        lines += ['| ' + ' | '.join(calibrated_columns) + ' |', '| ' + ' | '.join(['---'] * len(calibrated_columns)) + ' |']
        for row in calibrated_rows:
            lines.append('| ' + ' | '.join(str(row.get(c)) if row.get(c) is not None else 'undefined' for c in calibrated_columns) + ' |')
        lines += ['', 'Exports: `casia_calibrated_threshold_summary.csv/json`; per-run frame/video calibration JSON and train threshold curves; per-split `*_metrics_casia_calibrated.json` and `*_predictions_casia_calibrated.csv`.',
                  'Original source-frozen summaries, predictions and metrics are retained. Runtime configs are not overwritten.', '']
    else:
        lines += ['', 'CASIA threshold calibration disabled; source-frozen results only.', '']
    (OUTPUT_ROOT/'casia_train_test_evaluation_report.md').write_text('\n'.join(lines))
    archive_path=OUTPUT_ROOT.with_suffix('.zip')
    with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT_ROOT.rglob('*')):
            if path.is_file() and path.name!='shared_crops_bgr_uint8.npy' and not path.name.endswith('.partial.csv'):
                archive.write(path,arcname=path.relative_to(OUTPUT_ROOT))
    print('Saved report and ZIP:',archive_path,flush=True)
    print('Shared crop cache is excluded from the ZIP; raw detection/crop metadata is included.')
    display(FileLink(str(archive_path)))

    return summary,calibrated_rows


In [6]:
SCORE_SPLITS=('test','lcc_training','lcc_development','lcc_evaluation','lcc_combined')
EVALUATION_PROTOCOL={'objective':{'CE_clean':1.,'CE_worst':0.,'L_aux':0.},
 'train':'full eligible official CelebA Train','selection':'full official CelebA Test-as-Val; reused, not independent Test',
 'max_epochs':12,'milestones':[4,8],'patience_start_epoch':9,'early_stop_patience':3,
 'lcc':'full training/development/evaluation/combined; source threshold, no target fitting',
 'casia':'all extracted train/test color frames + deduplicated combined; notebook20 preprocessing',
 'casia_calibration':'train only, separate frame/video thresholds' if CALIBRATE_CASIA_THRESHOLD else 'disabled',
 'seed':TRAIN_SEED,'crop_factor':1.5,'comparison':'CE-only vs historical C (CE_clean + L_aux) and P3/R7; descriptive one-seed'}
write_json(out/'evaluation_protocol.json',EVALUATION_PROTOCOL)
def export_summary():
    rows=[]
    for name in RUNS:
        folder=out/'runs'/name
        if not (folder/'completion.json').exists():continue
        epoch=read_json(folder/'completion.json')['best_epoch']
        for split in SCORE_SPLITS:
            rows.append({'run_id':name,'dataset':split,'unit':'frame','selected_epoch':epoch,
                         'threshold_policy':'source-calibrated',**read_json(folder/'metrics'/f'{split}_metrics.json')})
        casia=folder/'casia'
        if (casia/'comparison_summary.json').exists():
            for row in read_json(casia/'comparison_summary.json'):
                rows.append({'run_id':name,'dataset':'casia_'+row['split'],'unit':'video','selected_epoch':epoch,
                             'threshold_policy':'source-calibrated',**{k:v for k,v in row.items() if not k.startswith('frame_')}})
                rows.append({'run_id':name,'dataset':'casia_'+row['split'],'unit':'frame','selected_epoch':epoch,
                             'threshold_policy':'source-calibrated',**{k[6:]:v for k,v in row.items() if k.startswith('frame_')}})
        if (casia/'casia_calibrated_threshold_summary.json').exists() and CALIBRATE_CASIA_THRESHOLD:
            for row in read_json(casia/'casia_calibrated_threshold_summary.json'):
                rows.append(row|{'dataset':'casia_'+row['split'],'selected_epoch':epoch,'threshold_policy':'CASIA train-calibrated'})
    pd.DataFrame(rows).to_csv(out/'comparison_summary.csv',index=False)
    write_json(out/'comparison_summary.json',rows)
    lines=['# CE-clean-only full-scale baseline','',
           'Loss = CE_clean only (weight1). No spectral selection, CE_worst, auxiliary forward or L_aux.',
           'Historical C_FULL used CE_clean + L_aux. Auxiliary modules are frozen compatibility placeholders; not optimized.',
           f'Matched notebook18: full eligible Train, crop1.5 BGR80, pretrained trunk/seed{TRAIN_SEED}, SGD .005/.9/5e-4, batch256 AMP, max12 epochs, milestones[4,8], gamma.2, phase-aware patience9/3. Checkpoints eligible from epoch1.',
           'CelebA full Test-as-Val is reused for selection and calibration, as in notebook18: not independent held-out Test evidence.',
           'LCC uses the source-frozen threshold. CASIA source and optional train-only calibrated thresholds are separate; video and frame units are separate.',
           'CASIA train and combined calibrated metrics include fitting data; Kaggle train/test copy overlap limits independence. No webcam validation or significance claim.','',
           '| Run | Dataset | Unit | Threshold policy | AUC | EER | APCER | BPCER | ACER | HTER | Scored | Candidates |',
           '|---|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|']
    columns=['run_id','dataset','unit','threshold_policy','auc','eer','apcer','bpcer','acer','hter','scored_n','candidate_n']
    for row in rows:lines.append('| '+' | '.join(str(row.get(k)) if row.get(k) is not None else 'undefined' for k in columns)+' |')
    lines+=['','Metrics are fractions. ACER=HTER=(APCER+BPCER)/2; Real iff logit(real)-logit(attack) >= threshold.',
            'Source and LCC predictions/metrics: runs/<run_id>/predictions and metrics. CASIA predictions, threshold curves, coverage and report: runs/<run_id>/casia.',
            'One new model, one seed. No new-model results exist until training/evaluation completes.','']
    (out/'ce_clean_fullscale_report.md').write_text('\n'.join(lines))
    if rows:display(pd.DataFrame(rows)[columns])
    return rows

def export_overall_zip():
    archive_path=out.with_suffix('.zip')
    with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for path in sorted(out.rglob('*')):
            if path.is_file() and path.name!='shared_crops_bgr_uint8.npy' and not path.name.endswith('.partial.csv'):
                archive.write(path,path.relative_to(out))
    print('Saved results ZIP:',archive_path,flush=True)
    display(FileLink(str(archive_path)))

def export_run_zip(name):
    folder=out/'runs'/name
    archive_path=out/f'{name}_s{TRAIN_SEED}.zip'
    with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for path in sorted(folder.rglob('*')):
            if path.is_file() and path.name!='shared_crops_bgr_uint8.npy' and not path.name.endswith('.partial.csv'):
                archive.write(path,path.relative_to(folder))
    print('Saved run ZIP:',archive_path,flush=True)
    display(FileLink(str(archive_path)))


## Train and evaluate

Progress prints from the first batch. RESUME continues epoch-boundary training; completed source/LCC evaluation is retained while CASIA runs separately. The ZIP excludes the shared CASIA crop array.


In [7]:
selected=select_runs(RUN_FILTER)
print('[Runner] Selected CE-only baseline:',selected,flush=True)
for name in selected:
    run_one(name,RUNS[name])
    export_summary()  # Save source/LCC results before CASIA detection/evaluation.
    folder=out/'runs'/name
    casia_done=folder/'casia_completion.json'
    if not casia_done.exists():
        best=torch.load(folder/'best.pth',map_location='cpu',weights_only=True)
        model,_=init_model({'init':'pretrained','copied':True})
        model.load_state_dict(best['state_dict'],strict=True);model=model.to(device).eval()
        locked=read_json(folder/'evaluation_freeze.json')['locked_threshold']
        print('[CASIA] Reloaded best checkpoint:',name,'epoch',best['epoch'],flush=True)
        evaluate_casia(model,folder,name,locked,best['epoch'])
        write_json(casia_done,{'run_id':name,'best_epoch':best['epoch'],'source_threshold':locked,
                              'train_threshold_calibration':CALIBRATE_CASIA_THRESHOLD,'status':'complete'})
        del model;gc.collect();torch.cuda.empty_cache()
    else:
        expected=read_json(casia_done)
        if expected['train_threshold_calibration']!=CALIBRATE_CASIA_THRESHOLD:raise RuntimeError('CASIA completion settings changed')
        casia=folder/'casia'
        needed=[casia/'comparison_summary.json',casia/'casia_train_test_evaluation_report.md']
        if CALIBRATE_CASIA_THRESHOLD:needed.append(casia/'casia_calibrated_threshold_summary.json')
        if not all(p.exists() for p in needed):raise RuntimeError('Incomplete CASIA outputs; remove only casia_completion.json to rerun CASIA')
        print('[CASIA] Already completed; skipping',flush=True)
    export_summary();export_overall_zip();export_run_zip(name)
print('Finished CE-only baseline:',selected,flush=True)


[Runner] Selected CE-only baseline: ['CE_CLEAN_FULL_crop15']
[CE_CLEAN_FULL_crop15] epoch 1/12 starting, warmup=False


/tmp/ipykernel_23/1895080082.py:403: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_23/1895080082.py:312: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[CE_CLEAN_FULL_crop15] epoch 1 batch 1/1893, loss=0.6240, elapsed=15.3s
[CE_CLEAN_FULL_crop15] epoch 1 batch 25/1893, loss=0.2980, elapsed=41.3s
[CE_CLEAN_FULL_crop15] epoch 1 batch 50/1893, loss=0.2182, elapsed=72.4s
[CE_CLEAN_FULL_crop15] epoch 1 batch 75/1893, loss=0.1838, elapsed=99.6s
[CE_CLEAN_FULL_crop15] epoch 1 batch 100/1893, loss=0.1654, elapsed=125.4s
[CE_CLEAN_FULL_crop15] epoch 1 batch 125/1893, loss=0.1558, elapsed=154.2s
[CE_CLEAN_FULL_crop15] epoch 1 batch 150/1893, loss=0.1468, elapsed=179.7s
[CE_CLEAN_FULL_crop15] epoch 1 batch 175/1893, loss=0.1402, elapsed=204.4s
[CE_CLEAN_FULL_crop15] epoch 1 batch 200/1893, loss=0.1346, elapsed=229.7s
[CE_CLEAN_FULL_crop15] epoch 1 batch 225/1893, loss=0.1301, elapsed=256.6s
[CE_CLEAN_FULL_crop15] epoch 1 batch 250/1893, loss=0.1268, elapsed=280.9s
[CE_CLEAN_FULL_crop15] epoch 1 batch 275/1893, loss=0.1231, elapsed=305.1s
[CE_CLEAN_FULL_crop15] epoch 1 batch 300/1893, loss=0.1182, elapsed=330.3s
[CE_CLEAN_FULL_crop15] epoch 1 bat

/tmp/ipykernel_23/1895080082.py:301: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))


[Inference] batch 25/260, elapsed=50.6s
[Inference] batch 50/260, elapsed=88.8s
[Inference] batch 75/260, elapsed=131.4s
[Inference] batch 100/260, elapsed=173.1s
[Inference] batch 125/260, elapsed=219.5s
[Inference] batch 150/260, elapsed=263.7s
[Inference] batch 175/260, elapsed=304.9s
[Inference] batch 200/260, elapsed=345.1s
[Inference] batch 225/260, elapsed=386.9s
[Inference] batch 250/260, elapsed=432.3s
[Inference] batch 260/260, elapsed=447.3s
CE_CLEAN_FULL_crop15 epoch 1 Full Test-as-Val ACER 0.0888329331272496 best 1 patience 0
[CE_CLEAN_FULL_crop15] epoch 2/12 starting, warmup=False
[CE_CLEAN_FULL_crop15] epoch 2 batch 1/1893, loss=0.0615, elapsed=3.2s


...

Saved results ZIP: /kaggle/working/21_fullscale_CE_clean_baseline_crop15_s105.zip


/kaggle/working/21_fullscale_CE_clean_baseline_crop15_s105.zip

Saved run ZIP: /kaggle/working/21_fullscale_CE_clean_baseline_crop15_s105/CE_CLEAN_FULL_crop15_s105.zip


/kaggle/working/21_fullscale_CE_clean_baseline_crop15_s105/CE_CLEAN_FULL_crop15_s105.zip

Finished CE-only baseline: ['CE_CLEAN_FULL_crop15']
